In [131]:
import numpy as np

In [129]:
def mini_batching(batch_size, X_train):
    batch = []
    batch_count = 0
    batch_last = []
    for i in range(len(X_train) // batch_size):
        batch.append(X_train[batch_size*i:batch_size*(i+1)])
        batch_count += 1
        print(len(X_train[batch_size*i:batch_size*(i+1)]))
    batch_last = X_train[batch_count*batch_size:]
    if len(batch_last) != 0:
        batch.append(batch_last)
    return batch, len(batch)

In [133]:
def initialize_parameters(layer_dims):
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l - 1]) * np.sqrt(2) / layer_dims[l - 1] 
        parameters['b' + str(l)] = np.zeros(layer_dims[l], 1)
    return parameters

In [134]:
def batch_norm_forward(Z, gamma, beta, epsilon = 1e-7):
    mu = np.mean(Z, axis = 1, keepdims = True)
    xmu = Z - mu
    var = np.mean(xmu**2, axis = 1, keepdims = True)
    sqrt_var = np.sqrt(var + epsilon)
    ivar = 1.0 / sqrt_var

    Z_norm = xmu * ivar

    Z_tilde = gamma * Z_norm + beta

    cache = (Z_norm, xmu, ivar, sqrt_var, var, mu, gamma)
    return Z_tilde, gamma

In [132]:
def forward_pass(X, parameters, batch_norm_forward):
    cache = {}
    cache['A0'] = X
    L = len(parameters) // 2
    for l in range(1, L):
        Z = np.dot(parameters['W' + str(l)], cache['A' + str(l - 1)]) + parameters['b' + str(l)]
        Z_tilde, b_cache = batch_norm_forward(Z, parameters['gamma' + str(l)], parameters['beta' + str(l)])
        cache['Z' + str(l)] = Z
        cache['Z_tilde' + str(l)] = Z_tilde
        cache['b_cache' + str(l)] = b_cache
        cache['A' + str(l)] = np.maximum(0, Z_tilde)
    ZL = np.dot(parameters['W' + str(l)], cache['A' + str(L)]) + parameters['b' + str(L)]
    cache['Z' + str(L)] = ZL
    cache['A' + str(L)] = 1 / (1 + np.exp(-Z))
    AL = cache['A' + str(L)]
    return AL, cache